In [ ]:

# Task 2 – Simple Graph-Based Image Segmentation
# This version also shows the NUMBER OF SEGMENTS in the output title.

import cv2              # OpenCV – for loading and processing images
import numpy as np      # NumPy – for working with arrays
import matplotlib.pyplot as plt  # Matplotlib – for showing images


def simple_graph_segmentation(image_path, theta=15):
    """Very simple graph-based segmentation using pixel intensity differences.

    Parameters
    ----------
    image_path : str
        File path to the input image.
    theta : int or float
        Threshold that controls how strongly pixels must be similar
        to be merged into the same segment. Smaller = more segments.

    Returns
    -------
    out_img : ndarray (H, W, 3)
        Colour image where each segment has a random colour.
    labels_2d : ndarray (H, W)
        2D array of segment labels for each pixel.
    num_segments : int
        Total number of distinct segments in the final result.
    """

    # 1) Load image from disk (OpenCV loads in BGR colour format)
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    # Get image height (h), width (w) and number of colour channels (_)
    h, w, _ = img_bgr.shape

    # Convert the colour image to grayscale (single intensity channel)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.int32)

    # Flatten 2D grayscale image (h x w) to a 1D array (h*w)
    flat_gray = gray.flatten()

    # 2) Each pixel is initially its own segment
    num_pixels = h * w
    seg = np.arange(num_pixels, dtype=np.int32)

    # 3) Build the list of graph edges between neighbouring pixels
    edges = []
    # 8-neighbourhood, but we only check 4 directions to avoid duplicates
    directions = [(1, 0), (0, 1), (1, 1), (-1, 1)]

    for y in range(h):
        for x in range(w):
            p = y * w + x
            for dx, dy in directions:
                nx = x + dx
                ny = y + dy
                if 0 <= nx < w and 0 <= ny < h:
                    q = ny * w + nx
                    # edge weight = absolute intensity difference
                    w_edge = abs(int(flat_gray[p]) - int(flat_gray[q]))
                    edges.append((w_edge, p, q))

    # 4) Sort edges by weight so that we connect most similar pixels first
    edges.sort(key=lambda e: e[0])

    # 5) Merge segments based on edge weights and threshold theta
    for weight, p, q in edges:
        if weight > theta:
            # treat as boundary
            continue

        if seg[p] != seg[q]:
            old_id = seg[q]
            new_id = seg[p]
            seg[seg == old_id] = new_id

    # 6) Compute final number of segments
    unique_ids = np.unique(seg)
    num_segments = len(unique_ids)
    print(f"Number of segments: {num_segments}")

    # Create a colour image where each segment has a random colour
    out_img = np.zeros_like(img_bgr)
    rng = np.random.default_rng(0)
    color_map = {
        sid: rng.integers(0, 256, size=3, dtype=np.uint8).tolist()
        for sid in unique_ids
    }

    for idx in range(num_pixels):
        y = idx // w
        x = idx % w
        out_img[y, x] = color_map[seg[idx]]

    # 7) Display original and segmented images side by side
    rgb_in = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    rgb_out = cv2.cvtColor(out_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 5))

    # Left: original image
    plt.subplot(1, 2, 1)
    plt.imshow(rgb_in)
    plt.title("Input image")
    plt.axis("off")  # no axes

    # Right: segmentation result – show theta AND number of segments
    plt.subplot(1, 2, 2)
    plt.imshow(rgb_out)
    plt.title(f"Segmentation (theta={theta}, segments={num_segments})")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # Return both the colourful output image, label map and segment count
    return out_img, seg.reshape(h, w), num_segments


# ================== Example usage in Google Colab ==================
# Steps:
# 1. Upload your image file to Colab (for example "plates.png" or "IMAGE 1.png").
# 2. Colab usually puts uploaded files in the /content folder.
# 3. Set 'example_path' below to the correct file name.
# 4. Run this cell to see the segmentation result and number of segments.

example_path = '/content/plates.png'  # change this if your file name is different
segmented_image, labels, n_segments = simple_graph_segmentation(example_path, theta=15)
print("Returned number of segments:", n_segments)
